[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q3/02_shap.ipynb)

# R1-Q3 Week 3 — Attribute the classifier's predictions

### This notebook produces an attribution table and per-class top-attributed gene lists.

This notebook applies the attribution method you precommitted to in Week 1 to the classifier you trained in Week 2, on the same held-out test split. For each prediction the classifier makes on the test set, the attribution method assigns every gene a score showing how much that gene pushed the prediction toward or away from the predicted class. Those scores, aggregated per class, are what you will compare against curated reference gene sets in Week 4 to assign the Strong, Mixed, or Low Overlap verdict.

By the end of this notebook you will have:

- An attribution table (per-sample, per-gene attribution scores from the precommitted method) saved as `attribution_scores.parquet`, ready for the Week 4 comparison notebook to load.
- Per-class top-attributed gene lists saved as `top_attributed_genes.json`, the headline result you will present in your Week 3 draft.
- An `attribution_summary.json` recording the attribution method, the classifier checkpoint it was applied to, and the parameters used, ready as input to your Week 3 draft presentation.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main
!pip install shap

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R1-Q3 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q3")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Load Notebook 01's outputs and check the gate

You wrote four artifacts to `OUTPUT_DIR` at the end of Notebook 01 — the pickled classifier(s), the held-out test sets, the control samples held back as a SHAP background, and the classifier summary JSON. Notebook 00 wrote a fifth: `precommit.json`. This section reads all five, cross-checks them against each other, and refuses to run if anything is missing or inconsistent.

Three reasons to load defensively rather than trust the files:

- **A missing file is cheap to catch here.** If `classifier_summary.json` is missing — because Notebook 01's Section 5 accuracy gate failed and refused to write it — Notebook 02 should stop in this cell rather than spend SHAP compute time on a model nobody verified.
- **Cross-checks catch silent drift.** The pickled `classifiers` dict and `classifier_summary["metadata"]["classifier_type"]` should agree on what kind of model is in the pickle. If they don't, something has been edited out of band — maybe Notebook 01 was re-run halfway with a different setup, maybe an output directory has files from two different sessions. Surfacing that here is much cheaper than chasing it down after SHAP runs.
- **The gate verdict drives the rest of the notebook.** Notebook 01 wrote `accuracy_gate.overall_verdict` as `"pass"` or `"partial"`, or didn't write the JSON at all (`"fail"`). On `"pass"`, every subset gets attributed in Section 3. On `"partial"`, only the subsets marked `"pass"` get attributed — the others are blocked because SHAP rankings on a sub-threshold classifier are unreliable. The list this cell builds — `subsets_to_attribute` — is the contract Section 3's main loop reads.

In [ ]:
import json
import pickle
import pandas as pd

# --- The five files this notebook depends on ---
# Four from Notebook 01, one from Notebook 00. All live in OUTPUT_DIR.

NB01_ARTIFACTS = {
    "classifier":          "classifier.pkl",
    "test_splits":         "test_splits.parquet",
    "controls_background": "controls_background.parquet",
    "classifier_summary":  "classifier_summary.json",
}
NB00_ARTIFACTS = {
    "precommit": "precommit.json",
}

# --- Check existence; collect ALL missing names before raising ---
# Reporting them all at once is more useful than failing on the first one.

missing = [
    fname
    for fname in {**NB00_ARTIFACTS, **NB01_ARTIFACTS}.values()
    if not (OUTPUT_DIR / fname).exists()
]
if missing:
    raise FileNotFoundError(
        f"Missing artifacts in {OUTPUT_DIR}: {missing}. "
        "Notebook 02 needs all of these. The most common cause is that "
        "Notebook 01's accuracy gate failed in Section 5 — in which case "
        "classifier_summary.json won't exist by design, and the right move "
        "is to revisit Notebook 01 rather than continue here."
    )

# --- Load each artifact ---

with open(OUTPUT_DIR / NB01_ARTIFACTS["classifier"], "rb") as f:
    classifiers = pickle.load(f)

test_splits_df = pd.read_parquet(OUTPUT_DIR / NB01_ARTIFACTS["test_splits"])
controls_df    = pd.read_parquet(OUTPUT_DIR / NB01_ARTIFACTS["controls_background"])

with open(OUTPUT_DIR / NB01_ARTIFACTS["classifier_summary"], "r") as f:
    classifier_summary = json.load(f)

with open(OUTPUT_DIR / NB00_ARTIFACTS["precommit"], "r") as f:
    precommit = json.load(f)

print("Loaded artifacts:")
print(f"  classifiers         : {sorted(classifiers.keys())} ({len(classifiers)} subset(s))")
print(f"  test_splits_df      : {test_splits_df.shape[0]:,} rows x {test_splits_df.shape[1]:,} cols")
print(f"  controls_df         : {controls_df.shape[0]:,} rows x {controls_df.shape[1]:,} cols")
print(f"  classifier_summary  : top-level keys = {sorted(classifier_summary.keys())}")
print(f"  precommit           : top-level keys = {sorted(precommit.keys())}")

# --- Cross-check: pickled classifier type matches what classifier_summary recorded ---

if "metadata" not in classifier_summary or "classifier_type" not in classifier_summary["metadata"]:
    raise KeyError(
        "classifier_summary.json is missing metadata.classifier_type. "
        "This usually means Notebook 01's Section 6 didn't complete. "
        "Re-run Notebook 01 before continuing."
    )

expected_type = classifier_summary["metadata"]["classifier_type"]
for key, clf in classifiers.items():
    actual_type = type(clf).__name__
    if actual_type != expected_type:
        raise ValueError(
            f"Type mismatch for classifiers[{key!r}]: "
            f"classifier_summary recorded {expected_type!r}, "
            f"but the pickle contains {actual_type!r}. "
            "Something has drifted between the saved summary and the saved model — "
            "the most likely cause is that OUTPUT_DIR holds files from two different runs."
        )
print(f"\nClassifier type cross-check: all {len(classifiers)} model(s) are {expected_type}. ✓")

# --- Build subsets_to_attribute by branching on the gate verdict ---

if "accuracy_gate" not in classifier_summary:
    raise KeyError(
        "classifier_summary.json has no accuracy_gate block. "
        "Notebook 01's Section 5 didn't run to completion."
    )

verdict = classifier_summary["accuracy_gate"]["overall_verdict"]

if verdict == "fail":
    # Notebook 01 should have raised before writing the JSON; this branch is a safety net.
    raise RuntimeError(
        "classifier_summary.json shows accuracy_gate.overall_verdict == 'fail'. "
        "Notebook 01 should have refused to write this file. Revisit Notebook 01 "
        "before continuing."
    )

per_subset_verdicts = classifier_summary["accuracy_gate"]["per_subset"]

if verdict == "pass":
    subsets_to_attribute = sorted(classifiers.keys())
    print(f"\nAll subsets passed the gate. Queued for attribution: {subsets_to_attribute}")
elif verdict == "partial":
    subsets_to_attribute = sorted(
        k for k, v in per_subset_verdicts.items() if v["verdict"] == "pass"
    )
    blocked = sorted(
        k for k, v in per_subset_verdicts.items() if v["verdict"] != "pass"
    )
    print(f"\nPartial verdict. "
          f"Queued for attribution: {subsets_to_attribute}.  "
          f"Blocked (sub-threshold): {blocked}.")
else:
    raise ValueError(
        f"Unexpected accuracy_gate.overall_verdict: {verdict!r}. "
        "Expected one of: 'pass', 'partial', 'fail'."
    )

# --- Belt-and-suspenders: every subset to attribute must exist in the pickle ---

for key in subsets_to_attribute:
    if key not in classifiers:
        raise KeyError(
            f"subsets_to_attribute includes {key!r}, but the classifiers pickle "
            f"holds only {sorted(classifiers.keys())}. Pickle and summary have drifted."
        )

You now have:

- `classifiers` — a dict of fitted models keyed by subset. Section 3's SHAP loop iterates this.
- `test_splits_df` — the held-out test sets with `subset`, `sample_id`, `stress_class`, and feature columns. Section 3 filters this by subset and runs SHAP on the result.
- `controls_df` — the control samples held out from training, same column shape as `test_splits_df`. Section 2 builds the SHAP background from this.
- `classifier_summary` — Notebook 01's full metadata dict, including the gate verdict and tissue handling choice (which Section 3 reads).
- `precommit` — Notebook 00's precommitments. Section 2 reads `precommit["attribution_method"]["method"]` to confirm SHAP is the committed branch.
- `subsets_to_attribute` — the list Section 3's main loop iterates. One or two entries depending on tissue handling and any per-subset gate failures.

Section 2 picks the SHAP background and confirms the attribution method.

### 2) Confirm the attribution method and configure SHAP

You committed to an attribution method in Notebook 00 — one of SHAP, integrated gradients, or permutation importance. This section confirms that choice against `precommit.json` and configures the SHAP explainer for Section 3's main run. If you committed to a method other than SHAP, this cell raises — the right way to switch methods is to revisit Notebook 00 and update your precommit there, not to edit anything mid-chain.

**Two configuration choices recorded here, both mentor-given.**

**`feature_perturbation = "tree_path_dependent"`.** SHAP's `TreeExplainer` has two modes for computing how a feature contributes to a prediction:

- `"tree_path_dependent"` (the SHAP default and what most published genomics work uses). Walks the trained trees to compute attributions; doesn't need background data. Fast.
- `"interventional"`. Uses a background dataset of held-out samples (your controls would be the natural choice) to compute conditional expectations. More theoretically sound when features are correlated — which gene expression is — but it's much slower because every sample's attribution requires repeated forward passes over the background.

For Colab compute on ~22K features and one or two classifiers, tree_path_dependent is the practical choice. The trade-off worth knowing about: tree_path_dependent's correctness rests on a feature-independence assumption that gene expression data violates. In practice the top-attributed genes are usually similar across the two modes; the differences show up in the magnitudes and in the long tail of weakly-attributed features. If you find a result in Week 4 that hinges on the exact magnitude or on a weak attribution, that's a flag to consider re-running with interventional mode and a small background subsample.

**Controls are loaded but not passed to `TreeExplainer`.** Because tree_path_dependent doesn't take a `data` argument, the controls held out in Notebook 01's Section 4.1 don't enter the SHAP computation directly. They're still in scope — Notebook 03's metadata variance-partitioning test will use them as a comparison distribution, and this section prints the per-subset counts for awareness — but they don't shape the attribution values themselves.

**What gets recorded.** This cell initializes `attribution_summary`, the dict that Sections 3, 4, and 6 will append to. The `setup` block is populated here; the other blocks fill in as we go.

In [ ]:
# --- Confirm the attribution method against your Notebook 00 precommit ---

ATTRIBUTION_METHOD = precommit["attribution_method"]["method"]

if ATTRIBUTION_METHOD == "SHAP":
    pass  # SHAP path — continue below.
elif ATTRIBUTION_METHOD in ("integrated_gradients", "permutation_importance"):
    raise NotImplementedError(
        f"Your precommit.json shows attribution_method.method == {ATTRIBUTION_METHOD!r}, "
        "but this notebook implements only the SHAP path. "
        "If your real intent is to use a different method, the right way to change it "
        "is to revisit Notebook 00 and re-do that precommitment with intention there — "
        "not by editing precommit.json mid-chain. If your intent was SHAP, re-running "
        "Notebook 00 will surface that and produce a precommit.json the rest of the "
        "chain can use."
    )
else:
    raise ValueError(
        f"Unexpected attribution_method.method: {ATTRIBUTION_METHOD!r}. "
        "Expected one of: 'SHAP', 'integrated_gradients', 'permutation_importance'."
    )

print(f"Attribution method: {ATTRIBUTION_METHOD} (confirmed against precommit.json).")

# --- TreeExplainer configuration ---

FEATURE_PERTURBATION = "tree_path_dependent"   # mentor-given; rationale in the markdown above

print(f"TreeExplainer feature_perturbation: {FEATURE_PERTURBATION}.")

# --- Background availability (informational; not passed to TreeExplainer in this mode) ---

print()
print("Control samples available per subset (for reference; not used as SHAP background "
      "in tree_path_dependent mode):")
for key in subsets_to_attribute:
    n = int((controls_df["subset"] == key).sum())
    print(f"  {key}: {n} control samples")

# --- Initialize attribution_summary; Sections 3, 4, 6 append to this dict ---

attribution_summary = {
    "setup": {
        "attribution_method":   ATTRIBUTION_METHOD,
        "feature_perturbation": FEATURE_PERTURBATION,
        "background_data_used": False,        # True only if you switch to interventional
        "background_size_per_subset": {
            key: int((controls_df["subset"] == key).sum())
            for key in subsets_to_attribute
        },
    },
    # "attribution_run" — populated by Section 3.
    # "top_attributions" — populated by Section 4.
    # "metadata" — populated by Section 6 closeout.
}

print()
print(f"attribution_summary initialized with setup block. Keys: {sorted(attribution_summary.keys())}.")

You now have:

- `ATTRIBUTION_METHOD` — the confirmed string `"SHAP"`. Section 3 branches on this only to the degree that it's already filtered by the raise above; future versions of the notebook with IG or permutation branches would read it again.
- `FEATURE_PERTURBATION` — `"tree_path_dependent"`. Section 3 passes this to every `TreeExplainer` it constructs.
- `attribution_summary` — the running settings dict. Section 3 will append per-subset compute stats and the sum-invariant check result; Section 4 will append your top-N choice; Section 6 will write the whole thing to `attribution_summary.json`.

Section 3 runs the per-subset SHAP attribution.

### 3) Run SHAP attribution per subset

This is the main compute. For each subset queued by Section 1, build a `TreeExplainer`, compute SHAP values on the test set, and store the result.

**What gets stored per subset.** The full 3D array of SHAP values, with axes `(sample, feature, class)`. Different downstream uses want different slices of this cube — Section 4's top-N extraction wants per-class rankings (aggregate across samples within a class), Section 6's per-sample output wants per-sample profiles for the true class (slice across features for a fixed (sample, class) pair), Notebook 03's metadata test wants something similar but framed against batch and processing date. Computing SHAP is the expensive step; storing the result once and slicing it many ways is cheap.

**Shape robustness.** Modern SHAP (roughly version 0.43 and later) returns a `shap.Explanation` object whose `.values` is a uniform 3D numpy array of shape `(n_samples, n_features, n_classes)`. Older SHAP returns a Python list of `n_classes` 2D arrays, each shape `(n_samples, n_features)`. The code below handles both — `np.stack` converts the list form into the 3D form, after which everything downstream is uniform.

**Predictions alongside attributions.** The loop also calls `clf.predict(X_test)` so each sample has both its true class (from `stress_class` in the test splits) and the model's predicted class. Section 6 needs both columns when writing the per-sample parquet, and both are useful for the EQ#1 writeup.

**What gets recorded to `attribution_summary`.** Per-subset compute time, sample count, feature count, class count. The sum invariant check writes its result two cells below.

In [ ]:
import shap
import numpy as np
import time

# Storage for per-subset SHAP results. Section 4 reads this for top-N extraction;
# Section 6 writes per-sample attribution profiles to parquet.
shap_results = {}

# Initialize the attribution_run block; per-subset stats accumulate below.
attribution_summary["attribution_run"] = {"per_subset": {}}

print(f"Running SHAP TreeExplainer on {len(subsets_to_attribute)} subset(s) "
      f"with feature_perturbation='{FEATURE_PERTURBATION}'.")
print()

for key in subsets_to_attribute:
    # Filter test_splits_df to this subset; split metadata from features.
    sub_df     = test_splits_df[test_splits_df["subset"] == key].copy()
    sample_ids = sub_df["sample_id"].values
    y_true     = sub_df["stress_class"].values
    X_test     = sub_df.drop(columns=["sample_id", "subset", "stress_class"])
    print(f"[{key}] X_test: {X_test.shape[0]} samples x {X_test.shape[1]} features")

    # Predict so each sample has both true_class and predicted_class downstream.
    clf    = classifiers[key]
    y_pred = clf.predict(X_test)

    # Build the explainer and compute attributions.
    print(f"[{key}]   running TreeExplainer...")
    t0           = time.time()
    explainer    = shap.TreeExplainer(clf, feature_perturbation=FEATURE_PERTURBATION)
    explanation  = explainer(X_test)
    shap_seconds = time.time() - t0

    # --- Multi-class shape robustness ---
    # Modern SHAP: explanation.values is ndarray with shape (n_samples, n_features, n_classes).
    # Older SHAP: explanation.values is a list of (n_samples, n_features) arrays, one per class.
    raw_values = explanation.values
    if isinstance(raw_values, list):
        shap_values = np.stack(raw_values, axis=-1)   # -> (samples, features, classes)
    else:
        shap_values = raw_values

    if shap_values.ndim != 3:
        raise ValueError(
            f"[{key}] Unexpected SHAP values shape: {shap_values.shape}. "
            "Expected 3D (n_samples, n_features, n_classes) for multi-class classification."
        )

    expected_value = np.array(explanation.expected_value)
    n_samples, n_features, n_classes_from_shap = shap_values.shape
    class_labels = list(clf.classes_)

    # Cross-check: SHAP's class count must match the classifier's.
    if n_classes_from_shap != len(class_labels):
        raise ValueError(
            f"[{key}] SHAP returned {n_classes_from_shap} classes but the classifier has "
            f"{len(class_labels)} classes ({class_labels}). Usually a SHAP version mismatch."
        )

    print(f"[{key}]   shap_values shape: {shap_values.shape}  "
          f"compute time: {shap_seconds:.1f}s")
    print()

    shap_results[key] = {
        "shap_values":    shap_values,         # (n_samples, n_features, n_classes)
        "expected_value": expected_value,      # (n_classes,)
        "feature_names":  list(X_test.columns),
        "class_labels":   class_labels,
        "sample_ids":     sample_ids,
        "y_true":         y_true,
        "y_pred":         y_pred,
    }

    attribution_summary["attribution_run"]["per_subset"][key] = {
        "n_samples":            int(n_samples),
        "n_features":           int(n_features),
        "n_classes":            int(n_classes_from_shap),
        "shap_compute_seconds": round(shap_seconds, 2),
    }

print(f"SHAP run complete. shap_results keys: {sorted(shap_results.keys())}.")

**SHAP sum invariant check.**

SHAP attributions satisfy a mathematical identity: for a given sample `i` and class `c`,

    expected_value[c] + sum over features of shap_values[i, :, c]  ≈  model.predict_proba(X[i])[c]

The base value (`expected_value[c]`) plus all the per-feature contributions should reconstruct the model's predicted probability for that class. This is the "explanation that adds up" property — it's what makes SHAP attributions interpretable as contributions rather than just rankings.

If this identity holds, `TreeExplainer` wired up correctly and the SHAP values it returned are usable downstream. If it doesn't hold — even approximately — something has gone wrong, usually one of:

- SHAP version mismatch with the installed scikit-learn or with the classifier type.
- `feature_perturbation` set to a value the explainer doesn't expect for this classifier.
- A model/pickle drift not caught by Section 1's classifier-type cross-check.

**This is a structural sanity check, not a gate.** A failure here doesn't make your attributions useless — the relative ordering of features may still be qualitatively correct, and a small numerical drift can come from floating-point arithmetic alone. The point is to surface anything that would cause silent confusion downstream. The cell below prints `pass` or `warn` per subset and writes the maximum absolute difference into `attribution_summary["attribution_run"]`, so Notebook 03 (and your EQ#1 writeup) can see what you saw. A warn doesn't block the rest of the notebook from running — you decide whether to investigate or accept the discrepancy.

The check runs on a random 20-sample subset (or all samples if fewer), with a fixed seed for reproducibility. That's enough to catch any structural problem without paying for a full check.

In [ ]:
INVARIANT_TOLERANCE = 1e-3   # max acceptable |predict_proba − (expected + sum(shap))|
INVARIANT_N_CHECK   = 20     # samples to spot-check per subset
INVARIANT_SEED      = 42     # fixed for reproducibility across runs

print("SHAP sum invariant check "
      f"(random {INVARIANT_N_CHECK}-sample subset per subset, seed={INVARIANT_SEED}):")
print()

all_passed = True

for key, result in shap_results.items():
    shap_values    = result["shap_values"]
    expected_value = result["expected_value"]
    n_samples      = shap_values.shape[0]

    # Re-derive X_test for predict_proba; same filtering used in Cell 3.B above.
    sub_df = test_splits_df[test_splits_df["subset"] == key]
    X_test = sub_df.drop(columns=["sample_id", "subset", "stress_class"])

    # SHAP-reconstructed probabilities per (sample, class):
    # expected_value[c]  +  sum over features of shap_values[i, :, c]
    shap_reconstructed = shap_values.sum(axis=1) + expected_value   # (n_samples, n_classes)

    # Sklearn's predict_proba on the same X_test, same row order.
    clf   = classifiers[key]
    proba = clf.predict_proba(X_test)                                # (n_samples, n_classes)

    # Spot-check a random subset of samples.
    n_check  = min(INVARIANT_N_CHECK, n_samples)
    rng      = np.random.default_rng(seed=INVARIANT_SEED)
    idx      = rng.choice(n_samples, size=n_check, replace=False)
    max_diff = float(np.abs(proba[idx] - shap_reconstructed[idx]).max())

    passed = max_diff < INVARIANT_TOLERANCE
    marker = "✓ pass" if passed else f"⚠ warn (max_diff exceeds {INVARIANT_TOLERANCE})"
    print(f"  {key}: max |proba − (expected + sum(shap))| over {n_check} samples "
          f"= {max_diff:.2e}   {marker}")

    attribution_summary["attribution_run"]["per_subset"][key]["sum_invariant_max_diff"] = max_diff
    attribution_summary["attribution_run"]["per_subset"][key]["sum_invariant_passed"]   = bool(passed)
    if not passed:
        all_passed = False

print()
if all_passed:
    print("All subsets pass the SHAP sum invariant. Attribution values are usable downstream.")
else:
    print("WARNING: SHAP sum invariant failed for at least one subset. "
          "Attribution values may be unreliable. Common causes: SHAP version mismatch, "
          "feature_perturbation setting that doesn't match the classifier's output space, "
          "or a model/pickle drift.")
    print("This warning does NOT block the rest of the notebook — you decide whether to "
          "investigate or accept the discrepancy.")

You now have:

- `shap_results` — a dict keyed by subset, each value a dict holding `shap_values` (3D ndarray of shape `(n_samples, n_features, n_classes)`), `expected_value` (per-class baseline output), `feature_names`, `class_labels`, `sample_ids`, `y_true`, `y_pred`. Section 4 aggregates `shap_values` across samples to produce per-class rankings; Section 6 writes per-sample attribution profiles to disk.
- `attribution_summary["attribution_run"]["per_subset"]` — compute time, sample/feature/class counts, sum-invariant max diff, sum-invariant pass/warn for each subset. Section 6 writes this to `attribution_summary.json`.

Section 4 extracts the top-attributed genes per stress class.

#### 4) — Extract top-attributed genes per class

For each (subset, class) combination, this section ranks the genes by how much they contributed to predictions of that class — then takes the top N. The top-N lists are one of the two outputs Notebook 03 reads. They feed the overlap test: do the most-attributed genes for "cold" overlap with known cold-stress genes from your reference set? If yes, the model learned biology; if no, the artifact-like rule trends toward "the model learned something else."

**You pick N here.** This is the one student methodological decision in Notebook 02. The choice is between a strict cutoff that admits only the strongest signals or a permissive cutoff that includes weaker but still real attributions. Different choices produce different verdicts in Notebook 03's overlap test, so this matters.

**A defensible range for N: 50 to 500.**

- **Top 50.** The strictest defensible cutoff. Only the strongest 50 attributions per class count; the overlap test becomes very specific (a few well-known stress genes have to appear or the verdict trends toward artifact-like). Good if you want to make confident "the model learned biology" claims — when the overlap test passes at N=50, it's because the model latched onto known stress markers. The cost is that the test fails for any model that uses a wider set of biologically relevant genes whose individual contributions are weaker.
- **Top 100-200.** The middle range. Most published SHAP-on-genomics work uses something in this band. Catches the strongest stress markers plus a reasonable tail of secondary genes. Less strict, less easy to fool by a model that uses a narrow shortcut.
- **Top 500.** The most permissive defensible cutoff. The overlap test gets forgiving — even a model using a broad mix of stress-correlated and stress-uncorrelated genes will likely show some overlap. The cost is interpretive: a passing verdict at N=500 doesn't tell you whether the model is using biology or whether the dataset is just dense with stress-relevant variation.
- **Below 50 or above 500.** Defensible only with a specific reason. If you can argue for top 20 because you want to test whether the model latched onto a known small regulon, fine. If you can argue for top 1000 because you're characterizing a broad transcriptional response, fine. Otherwise stick in the 50-500 band.

**Pre-commit before you see the result.** Cell 4.C asks you to write down `TOP_N` and a rationale. Cell 4.E then runs the aggregation and prints a preview of the top 10 genes per class for the first subset. If you change `TOP_N` after seeing the preview, you're fitting your cutoff to your result — the cutoff stops being a choice you made and becomes a number that produced the answer you wanted. The notebook can't prevent you from doing this; the cell ordering is the prompt to do it honestly.

Your `TOP_N` and rationale go into `attribution_summary["top_attributions"]` and your Week 4 EQ#1 writeup.

In [ ]:
# Commit your top-N cutoff and rationale BEFORE running Cell 4.E.
# Cell 4.E prints a preview of the top 10 genes per class; if you change TOP_N
# after seeing it, the cutoff stops being a real choice and becomes a number
# that produced an answer you wanted.

TOP_N           = ...    # an integer between 50 and 500 (or outside with a strong reason)
TOP_N_RATIONALE = ...    # 1-3 sentences: why this cutoff for your data and your overlap test ambitions

# --- Validation ---

if not isinstance(TOP_N, int):
    raise ValueError(
        f"TOP_N must be an integer; got {type(TOP_N).__name__}. "
        "Edit Cell 4.C and pick an integer (50-500 is the defensible band)."
    )
if TOP_N < 1:
    raise ValueError(
        f"TOP_N is {TOP_N}; must be a positive integer."
    )

# The number of features available constrains the maximum. The smallest subset
# wouldn't have enough features to rank top-N if N exceeds its feature count.
min_n_features = min(
    attribution_summary["attribution_run"]["per_subset"][key]["n_features"]
    for key in subsets_to_attribute
)
if TOP_N > min_n_features:
    raise ValueError(
        f"TOP_N is {TOP_N}, but the smallest subset has only {min_n_features} features. "
        "Pick a smaller value."
    )

if not isinstance(TOP_N_RATIONALE, str) or not TOP_N_RATIONALE.strip() or TOP_N_RATIONALE.strip() == "...":
    raise ValueError(
        "TOP_N_RATIONALE is empty or a placeholder. Write 1-3 sentences explaining "
        "your cutoff before continuing to Cell 4.E."
    )

# Soft notice for values outside the 50-500 defensible band — not a block.
if TOP_N < 50:
    print(f"NOTE: TOP_N = {TOP_N} is below the 50-500 defensible band. "
          "Make sure your rationale explains the specific reason.")
elif TOP_N > 500:
    print(f"NOTE: TOP_N = {TOP_N} is above the 50-500 defensible band. "
          "Make sure your rationale explains the specific reason.")

# Record into attribution_summary so Section 6 writes it to disk and Notebook 03 reads it.
attribution_summary["top_attributions"] = {
    "top_n":              int(TOP_N),
    "top_n_rationale":    TOP_N_RATIONALE.strip(),
    "aggregation_method": "mean_abs",   # mentor-given; rationale in Cell 4.D below
}

print(f"TOP_N committed: {TOP_N}")
print(f"Rationale: {TOP_N_RATIONALE.strip()}")
print()
print("Run Cell 4.E to compute the rankings.")

**Aggregation: mean of absolute SHAP values across the test set.**

For each (subset, class) combination, the next cell collapses the `(n_samples, n_features)` slice of the 3D SHAP array into a length-`n_features` vector by taking the mean of absolute values across all test samples:

    score[gene] = mean over all test samples of |shap_value[sample, gene, class]|

This is the standard convention in the SHAP literature — `shap.summary_plot` uses the same aggregation when it shows per-class feature importance. Aggregating across all test samples (rather than restricting to samples whose true class is `class`) keeps the per-class scores stable; with 8-9 classes and 25-75 test samples per subset, per-class in-class counts can be as small as 2-5 samples, which is too few to give stable mean-abs scores.

Two alternatives worth knowing about and ruling out:

- **Mean of signed values.** Averages signed SHAP values without taking absolute value. Useful when you care whether a gene pushes predictions toward or away from a class — but it ranks badly. A gene that contributes +0.5 for half the samples and −0.5 for the other half averages to zero and disappears from the ranking entirely, even though it's clearly an important feature. Useful for direction interpretation; bad for importance ranking.
- **Sum of absolute values.** Sums `|shap|` instead of averaging. The ranking is identical to mean-abs up to a constant (the number of samples), but sum-abs grows with sample count, so cross-subset comparison gets harder. Mean-abs is the more common reporting convention.

The aggregation rule is recorded as `"mean_abs"` in `attribution_summary["top_attributions"]["aggregation_method"]`. If a future analysis wants to switch (say, to mean-signed for a directional question), Cell 4.E is the one place to change.

After aggregation, the next cell ranks each (subset, class) by descending `mean_abs_shap` and takes the top N. The result is a long-format DataFrame — one row per (subset, class, gene_id, rank, mean_abs_shap) — that Section 6 writes to `top_attributions.parquet`.

In [ ]:
# Aggregate SHAP values across the test set within each class, then rank.
# For each subset and each class, produce a top-N list of genes ordered by
# mean absolute SHAP value (descending).

records = []   # one row per (subset, class, gene_id, rank)

for key, result in shap_results.items():
    shap_values   = result["shap_values"]    # (n_samples, n_features, n_classes)
    feature_names = result["feature_names"]  # length n_features
    class_labels  = result["class_labels"]   # length n_classes
    y_true        = result["y_true"]

    for c_idx, class_label in enumerate(class_labels):
        # Defensive: surface if a class is absent from the test set entirely.
        # (Stratified split in NB01 should prevent this, but if it happens the
        # aggregation still runs — the score reflects cross-class signal only.)
        in_class_count = int((y_true == class_label).sum())
        if in_class_count == 0:
            print(f"  NOTE: no test samples with true class {class_label!r} in subset {key!r}. "
                  "Aggregation continues over the full test set.")

        # Mean-abs over ALL test samples, using the class-c slice of the 3D array.
        class_shap = shap_values[:, :, c_idx]                # (n_samples, n_features)
        mean_abs   = np.abs(class_shap).mean(axis=0)         # (n_features,)

        # Rank features by descending mean_abs_shap; take the top N.
        top_idx = np.argsort(mean_abs)[::-1][:TOP_N]         # indices into feature_names

        for rank, idx in enumerate(top_idx, start=1):
            records.append({
                "subset":        key,
                "class":         class_label,
                "gene_id":       feature_names[idx],
                "rank":          rank,
                "mean_abs_shap": float(mean_abs[idx]),
            })

top_attributions_df = pd.DataFrame(records)

print(f"Built top_attributions_df: {len(top_attributions_df):,} rows.")
print(f"  Subsets : {sorted(top_attributions_df['subset'].unique())}")
print(f"  Classes : {sorted(top_attributions_df['class'].unique())}")
print(f"  Top N   : {TOP_N}")
print()

# Preview: top 10 genes per class for the first subset (a sanity glance,
# not the final result). The full ranking is in top_attributions_df.
preview_subset = sorted(top_attributions_df["subset"].unique())[0]
preview_df = (
    top_attributions_df[top_attributions_df["subset"] == preview_subset]
    .sort_values(["class", "rank"])
    .groupby("class", group_keys=False)
    .head(10)
)
print(f"Preview — top 10 attributions per class for subset '{preview_subset}':")
print(preview_df.to_string(index=False))

You now have:

- `top_attributions_df` — a long-format DataFrame, one row per (subset, class, gene_id, rank), with `mean_abs_shap` as the score. Section 6 writes this to `top_attributions.parquet` for Notebook 03's overlap test.
- `attribution_summary["top_attributions"]` — your `top_n` value, your `top_n_rationale`, and the aggregation method. Section 6 writes the full `attribution_summary` to JSON.

Section 5 (optional) generates SHAP summary plots for the Week 3 draft presentation; Section 6 writes the final artifacts to disk.

### 5) Generate SHAP summary plots (optional)

This section produces PNG figures for your Week 3 draft presentation. Skip it on a development pass — Notebook 03 doesn't read these files. Run it once your `top_attributions_df` looks sensible and you're ready to write the presentation.

**What gets auto-generated:** one multi-class bar plot per subset. The plot ranks the top 20 features by mean absolute SHAP value across the test set, with each bar stacked or grouped by class so you can see which classes a given gene contributes to most. This is the "what features did the model rely on overall" view — the one most papers show on a methods slide.

**What doesn't:** per-class beeswarm plots (the dense scatter plots that show how each feature's value affects each sample's prediction). These are richer but expensive at 8-9 classes per subset, and most don't end up on a 10-slide presentation. Cell 5.C below gives you a template you can adapt for any specific class you want to dig into.

**Plot files land in `OUTPUT_DIR/plots/`.** That subdirectory is created if it doesn't exist. Naming convention: `{subset}_overview.png`. The plots aren't tracked in `attribution_summary.json` — they're presentation artifacts, separate from the data contract with Notebook 03.

**A note on tissue-as-covariate mode.** If you picked tissue-as-covariate back in Notebook 01 Section 3, the tissue one-hot columns are features in the model's eyes, and they'll appear in the summary plot alongside genes. That's informative — if `tissue_shoot` or `tissue_root` shows up in the top 20, it tells you the model is using tissue identity as a substantial predictor. Notebook 03's metadata test will say the same thing more formally; the plot is the eyeball version.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

MAX_DISPLAY_FEATURES = 20    # top 20 features shown per plot
PLOT_DPI             = 150   # presentation-quality

print(f"Generating SHAP summary plots in {PLOTS_DIR}...")
print()

saved_plots = []

for key in subsets_to_attribute:
    result        = shap_results[key]
    shap_values   = result["shap_values"]      # (n_samples, n_features, n_classes)
    feature_names = result["feature_names"]
    class_labels  = result["class_labels"]

    # Re-derive X_test for the plot. SHAP uses feature values to color bars
    # and to label axes; passing X_test lets shap.summary_plot check shape consistency.
    sub_df = test_splits_df[test_splits_df["subset"] == key]
    X_test = sub_df.drop(columns=["sample_id", "subset", "stress_class"])

    # shap.summary_plot's multi-class interface expects a list of (n_samples, n_features)
    # arrays — one per class — rather than the 3D form. Convert here.
    shap_values_list = [shap_values[:, :, c] for c in range(shap_values.shape[2])]

    # Multi-class summary bar plot.
    shap.summary_plot(
        shap_values_list,
        X_test,
        plot_type    = "bar",
        class_names  = class_labels,
        max_display  = MAX_DISPLAY_FEATURES,
        show         = False,             # important: stop SHAP from calling plt.show()
    )
    plt.tight_layout()

    plot_path = PLOTS_DIR / f"{key}_overview.png"
    plt.savefig(plot_path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close("all")                       # free memory and reset state

    saved_plots.append(plot_path)
    print(f"  [{key}]  saved {plot_path.name}  ({MAX_DISPLAY_FEATURES} top features, "
          f"{len(class_labels)} classes)")

print()
print(f"Saved {len(saved_plots)} overview plot(s) to {PLOTS_DIR}:")
for p in saved_plots:
    print(f"  {p}")

**Per-class beeswarms (do-it-yourself).** If a specific class shows up as interesting in your top-N analysis — for example, the overlap test is going to hinge on whether the model is using known cold-stress markers — a per-class beeswarm plot tells you a richer story than the bar. Each dot is a test sample, positioned by its SHAP value for the chosen class, colored by the gene's expression level. You can see whether high-expression samples push toward the class or away from it.

Template to copy into a new cell and adapt:

```python
# Per-class beeswarm template — change the two values and run.
CLASS_TO_PLOT  = "cold"     # one of the labels in result["class_labels"]
SUBSET_TO_PLOT = "shoot"    # one of subsets_to_attribute

result        = shap_results[SUBSET_TO_PLOT]
class_idx     = result["class_labels"].index(CLASS_TO_PLOT)
class_shap    = result["shap_values"][:, :, class_idx]

sub_df = test_splits_df[test_splits_df["subset"] == SUBSET_TO_PLOT]
X_test = sub_df.drop(columns=["sample_id", "subset", "stress_class"])

shap.summary_plot(class_shap, X_test, max_display=20, show=False)
plt.tight_layout()
beeswarm_path = PLOTS_DIR / f"{SUBSET_TO_PLOT}_{CLASS_TO_PLOT}_beeswarm.png"
plt.savefig(beeswarm_path, dpi=PLOT_DPI, bbox_inches="tight")
plt.close("all")
print(f"Saved beeswarm: {beeswarm_path}")
```

You now have:

- `OUTPUT_DIR/plots/{subset}_overview.png` — one bar plot per subset, top 20 features ranked by mean absolute SHAP across all classes.
- A beeswarm template for any class you want to dig into.

Section 6 writes the data artifacts (`attribution_scores.parquet`, `top_attributed_genes.json`, `attribution_summary.json`) and prints the EQ#2 checklist.

### 6) Closeout: save artifacts for Notebook 03 and the Week 3 draft

This section saves three files to `OUTPUT_DIR`. Together they're the contract with Notebook 03 and the data input to your Week 3 draft presentation.

- **`attribution_scores.parquet`** — per-sample attribution profiles, one row per (subset, sample_id). Columns: `subset`, `sample_id`, `true_class`, `predicted_class`, then one column per feature (genes plus tissue one-hot columns if you went tissue-as-covariate). The values are SHAP attributions for each sample's **true class** — what genes the model used when it saw a sample of that class. Notebook 03's metadata test partitions the variance of these profiles against array batch, processing date, and tissue identity.
- **`top_attributed_genes.json`** — top-N gene lists per (subset, class), keyed in a nested structure (`{subset: {class: [{gene_id, rank, mean_abs_shap}, ...]}}`). The mean-abs SHAP scores are included alongside the gene IDs so Notebook 03 can run weighted overlap tests if it wants to, and so you can spot-check the top genes in a text editor while writing your draft. JSON rather than parquet because the data is small, the structure is naturally hierarchical, and human-readable matters more here than column-oriented compression.
- **`attribution_summary.json`** — the running settings dict (`setup`, `attribution_run`, `top_attributions`, `metadata`). Notebook 03 reads this to know what method, what configuration, and what your TOP_N choice was. Your Week 3 draft pulls headline numbers from it.

**What's not saved.** The full 3D SHAP array per subset — samples × features × *all classes*. It's recoverable: with the same classifier, the same test set, and the same `feature_perturbation`, re-running Section 3 produces the same values. Storing all classes' attributions per sample would multiply the disk footprint of `attribution_scores.parquet` by the class count for almost no operational benefit — Notebook 03 only reads the true-class slice.

**True-class vs predicted-class attribution.** The per-sample parquet uses SHAP values for each sample's **true** class. The question the metadata test is asking — "what genes did the model use to predict cold stress" — assumes we're looking at the model's reasoning for samples that are actually cold. Using predicted-class attribution instead would shift the question to "what did the model use when it called something cold" — a different question, and not what Notebook 03 is set up for.

In [ ]:
# --- Build the per-sample attribution profile DataFrame ---
# Wide format: one row per (subset, sample_id). Columns: subset, sample_id,
# true_class, predicted_class, then one column per feature. Values are SHAP
# attributions for each sample's TRUE class.

attribution_rows = []

for key, result in shap_results.items():
    shap_values   = result["shap_values"]      # (n_samples, n_features, n_classes)
    feature_names = result["feature_names"]
    class_labels  = result["class_labels"]
    sample_ids    = result["sample_ids"]
    y_true        = result["y_true"]
    y_pred        = result["y_pred"]

    # Defensive: every true-class label must exist in the classifier's known classes.
    # (Would only fail if NB01 trained on classes that aren't in the test set.)
    unknown = set(y_true) - set(class_labels)
    if unknown:
        raise ValueError(
            f"[{key}] y_true contains labels not in classifier.classes_: {unknown}. "
            "Test split shouldn't carry classes the classifier didn't train on."
        )

    # For each sample, pick the SHAP column corresponding to its true class.
    # Fancy indexing across the sample and class axes; the slice keeps the feature axis whole.
    n_samples = shap_values.shape[0]
    class_idx_per_sample = np.array([class_labels.index(c) for c in y_true])
    true_class_shap = shap_values[np.arange(n_samples), :, class_idx_per_sample]
    # true_class_shap shape: (n_samples, n_features)

    # Assemble as a DataFrame with metadata columns up front.
    sample_df = pd.DataFrame(true_class_shap, columns=feature_names, index=sample_ids)
    sample_df.insert(0, "predicted_class", y_pred)
    sample_df.insert(0, "true_class",      y_true)
    sample_df.insert(0, "subset",          key)
    sample_df.index.name = "sample_id"

    attribution_rows.append(sample_df.reset_index())

attribution_scores_df = pd.concat(attribution_rows, ignore_index=True)

# Re-order columns so metadata comes first, then features. (concat preserves
# the per-subset column order which is already correct, but explicit ordering
# protects against per-tissue mode where the two subsets may have differing
# feature subsets in theory.)
metadata_cols = ["subset", "sample_id", "true_class", "predicted_class"]
feature_cols  = [c for c in attribution_scores_df.columns if c not in metadata_cols]
attribution_scores_df = attribution_scores_df[metadata_cols + feature_cols]

# --- Write ---

attribution_path = OUTPUT_DIR / "attribution_scores.parquet"
attribution_scores_df.to_parquet(attribution_path, index=False)

size_mb = attribution_path.stat().st_size / (1024 * 1024)
print(f"Wrote {attribution_path.name} ({size_mb:,.2f} MB).")
print(f"  Shape:    {attribution_scores_df.shape[0]:,} samples x "
      f"{attribution_scores_df.shape[1]:,} columns")
print(f"  Subsets:  {sorted(attribution_scores_df['subset'].unique())}")
print(f"  Features: {len(feature_cols):,} (genes + any tissue one-hot columns)")

In [ ]:
# --- Build the nested top_attributed_genes dict from top_attributions_df ---
# Structure: {subset: {class: [{"gene_id": ..., "rank": ..., "mean_abs_shap": ...}, ...]}}
# Each per-class list is ordered by rank ascending (rank 1 is the top-attributed gene).

top_attributed_genes = {}

for (subset, class_label), group in top_attributions_df.groupby(["subset", "class"]):
    sorted_group = group.sort_values("rank")
    top_attributed_genes.setdefault(subset, {})[class_label] = [
        {
            "gene_id":       row["gene_id"],
            "rank":          int(row["rank"]),
            "mean_abs_shap": float(row["mean_abs_shap"]),
        }
        for _, row in sorted_group.iterrows()
    ]

# --- Write ---

top_genes_path = OUTPUT_DIR / "top_attributed_genes.json"
with open(top_genes_path, "w") as f:
    json.dump(top_attributed_genes, f, indent=2)

# --- Quick sanity stats ---

total_entries = sum(
    len(genes)
    for subset_dict in top_attributed_genes.values()
    for genes in subset_dict.values()
)
n_subsets = len(top_attributed_genes)
sample_subset = next(iter(top_attributed_genes.values()))
n_classes_per_subset = len(sample_subset)

size_kb = top_genes_path.stat().st_size / 1024
print(f"Wrote {top_genes_path.name} ({size_kb:,.1f} KB).")
print(f"  Subsets:             {sorted(top_attributed_genes.keys())}")
print(f"  Classes per subset:  {n_classes_per_subset}")
print(f"  Top-N per class:     {TOP_N}")
print(f"  Total gene entries:  {total_entries:,}")

In [ ]:
from datetime import datetime, timezone

# --- Append closeout metadata to attribution_summary ---
# This block is the fourth top-level key; the first three (setup, attribution_run,
# top_attributions) were filled by Sections 2, 3, 4.

attribution_summary["metadata"] = {
    "notebook_completed_at": datetime.now(timezone.utc).isoformat(),
    "irilab2026_version":    iri.__version__,
    "notebook":              "02_shap.ipynb",
    "classifier_type":       classifier_summary["metadata"]["classifier_type"],
    "n_subsets_attributed":  len(subsets_to_attribute),
}

# --- Write attribution_summary.json ---

summary_path = OUTPUT_DIR / "attribution_summary.json"

# default=str catches anything that isn't JSON-native (datetime, numpy scalars
# that may have leaked through, Path objects, etc.) and converts to string.
with open(summary_path, "w") as f:
    json.dump(attribution_summary, f, indent=2, default=str)

# --- Read-back verification ---

with open(summary_path, "r") as f:
    summary_check = json.load(f)

expected_top_keys = {"setup", "attribution_run", "top_attributions", "metadata"}
missing = expected_top_keys - set(summary_check.keys())
if missing:
    raise RuntimeError(
        f"attribution_summary.json round-trip missing keys: {sorted(missing)}. "
        "Check that Sections 2, 3, 4, and 6 all completed."
    )

size_kb = summary_path.stat().st_size / 1024
print(f"Wrote {summary_path.name} ({size_kb:,.1f} KB).")
print(f"  Top-level keys: {sorted(summary_check.keys())}")
print()

# --- Final roll-up across all three artifacts ---

print("All three artifacts written to OUTPUT_DIR:")
artifact_names = [
    "attribution_scores.parquet",
    "top_attributed_genes.json",
    "attribution_summary.json",
]
for fname in artifact_names:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        size = fpath.stat().st_size
        size_str = f"{size / (1024*1024):.2f} MB" if size > 1024*1024 else f"{size / 1024:.1f} KB"
        print(f"  ✓ {fname:<32s} ({size_str})")
    else:
        print(f"  ✗ {fname:<32s} (MISSING)")

### Week 3 draft presentation — what goes in

The Week 3 deliverable is a draft presentation (slides, not paper). This is a checklist for it, pointing to where every number, list, and plot lives in the files you just wrote.

**Methods.** What attribution method, what classifier, what configuration.

- Attribution method: `attribution_summary["setup"]["attribution_method"]` (will read `"SHAP"`).
- TreeExplainer mode: `attribution_summary["setup"]["feature_perturbation"]` (will read `"tree_path_dependent"`). The trade-off against `"interventional"` is a one-sentence methodological choice worth mentioning if the slide has room.
- Classifier: from Notebook 01. Type is in `classifier_summary["metadata"]["classifier_type"]`; per-subset balanced accuracy is in `classifier_summary["evaluation"]["per_subset"][key]["balanced_accuracy"]`. Quote at least the headline number for the subset you'll focus on.
- Subsets attributed: from `subsets_to_attribute`. One (tissue-as-covariate) or two (per-tissue).

**Top-N cutoff.** Yours, recorded in `attribution_summary["top_attributions"]["top_n"]`. Your rationale is in the same block. Why this number, in one sentence — that's the choice that affects how strict your Week 4 overlap verdict will be.

**Results.** The headline visualization and the lists.

- Overview bar plot from Section 5: `OUTPUT_DIR/plots/{subset}_overview.png`. The methods or results slide.
- Top-attributed genes per class: `top_attributed_genes.json`. Pick one or two classes whose top genes you'll highlight. The slide doesn't need to show all 8-9 classes — pick the ones with the most interesting story (most well-known stress markers; or a class where the model is using something unexpected).
- Optional: per-class beeswarm for a class you want to dig into, generated via the template in Cell 5.C.

**What's coming in Week 4.** A one-slide preview of the artifact-like rule and its three possible verdicts: **Strong Overlap** (top-attributed genes match known stress markers well), **Mixed Overlap** (partial match), or **Low Overlap** (the model is not using known biology). Notebook 03 will run this comparison; this draft sets up the data.

- Reference gene sets: `precommit["reference_gene_set"]` — what known stress markers you precommitted to comparing against.
- Artifact-like definition: `precommit["artifact_like_definition"]` — the rule that turns the comparison into a verdict.
- The metadata test: not visible until Notebook 03 runs, but the input is your `attribution_scores.parquet`.

**Sanity check before you present.** Did the SHAP sum invariant pass on every subset? Look at `attribution_summary["attribution_run"]["per_subset"][key]["sum_invariant_passed"]`. If any subset is `False`, the SHAP values it produced may not have summed cleanly to predicted probabilities — the presentation should either explain why or skip findings from that subset.